<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/ML_Project_IPL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [40]:
pd.set_option('display.max_columns', None)
df_main = pd.read_csv('/content/IPL.csv',low_memory=False)


In [41]:
cols_remove = ['player_of_match','match_won_by','win_outcome','city','day','month','event_match_no','match_type','match_number','date','ball_no','fielders','team_reviewed','result_type','stage','review_batter','event_name','review_decision','umpire','umpires_call','season','gender','team_type','superover_winner','method','balls_per_over','overs','power_surge_start','wicket_kind','extra_type','new_batter','runs_not_boundary','next_batter','batting_partners']
df_main.drop(columns = cols_remove, inplace=True)

In [42]:
rename_cols = {
    "runs_batter" : "ball_runs_batter",
    "balls_faced" : "binary_ball_faced",
    "runs_bowler" : "ball_runs_bowler",
    "batter_runs" : "cum_batter_runs",
    "batter_balls" : "cum_batter_balls",
    "bowler_wicket" : "binary_bowler_wicket",
    "runs_extras" : "ball_runs_extras",
    "runs_total" : "ball_runs_total",
    "team_runs" : "cum_team_runs",
    "team_balls" : "cum_team_balls",
    "team_wicket" : "cum_team_wickets"
}

df_main = df_main.rename(columns = rename_cols)

In [43]:
df_main.drop(df_main[df_main['ball']==7].index,inplace=True)

In [44]:
df_main = df_main.sort_values(
    by=["match_id", "innings", "over", "ball"]
).reset_index(drop=True)

In [45]:
df_main["cum_bowler_runs"] = df_main.groupby(["match_id", "innings", "bowler"])[
    "ball_runs_bowler"
].cumsum()

df_main["cum_bowler_wickets"] = df_main.groupby(["match_id", "innings", "bowler"])[
    "binary_bowler_wicket"
].cumsum()

df_main["cum_bowler_balls"] = df_main.groupby(["match_id", "innings", "bowler"])[
    "valid_ball"
].cumsum()

In [46]:
team_cum_cols = ["cum_team_runs", "cum_team_balls", "cum_team_wickets"]
for col in team_cum_cols:
    df_main[col] = df_main.groupby(["match_id", "innings"])[col].shift(1).fillna(0)

# 3. Shift Batter-Level Cumulative Features (Grouped by match_id, innings, batter)
batter_cum_cols = ["cum_batter_runs", "cum_batter_balls"]
for col in batter_cum_cols:
    df_main[col] = (
        df_main.groupby(["match_id", "innings", "batter"])[col].shift(1).fillna(0)
    )

# 4. Shift Bowler-Level Cumulative Features (Grouped by match_id, innings, bowler)
bowler_cum_cols = ["cum_bowler_runs", "cum_bowler_wickets", "cum_bowler_balls"]
for col in bowler_cum_cols:
    df_main[col] = (
        df_main.groupby(["match_id", "innings", "bowler"])[col].shift(1).fillna(0)
    )

In [53]:
conditions = [
    df_main["player_out"].notna(),
    df_main["ball_runs_total"] == 5,
    df_main["ball_runs_total"] == 7,
]

choices = ["Wicket", "4", "6"]

# 2. Assign target variable (formatted as string class labels)
df_main["target"] = np.select(
    conditions, choices, default=df_main["ball_runs_total"].astype(str)
)

In [76]:
from google import genai
import json
import re
import pandas as pd
from google.colab import userdata
import time

# Retrieve the secret value
api_key = userdata.get('GEM_API')

client = genai.Client(api_key=api_key)


def llm_api(prompt):
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
    )
    return response.text

unique_players = list(
    set(df_main["batter"].dropna().unique()).union(
        set(df_main["bowler"].dropna().unique())
    )
)

PROMPT_TEMPLATE = """
You are a cricket database engine. Below is a list of cricket player names. For each player, identify their primary batting hand and primary bowling style.

Output ONLY a valid JSON array of objects with the exact schema:
[{"name": "Player Name", "batting_hand": "RHB" or "LHB", "bowling_style": "RA_Pace" or "LA_Pace" or "RA_Offspin" or "LA_Orthodox" or "Wrist_Spin" or "None"}]

Rules:
1. Use "None" for bowling_style if the player rarely or never bowls in T20s.
2. Resolve abbreviated names (e.g., 'V Kohli' -> RHB, RA_Pace).
3. If a player is completely unknown, make a reasonable guess or set bowling_style to "None".

Player List:
{batch_players}
"""


def generate_player_metadata_robust(
    player_list, batch_size=100, delay_seconds=12
):
    batches = [
        player_list[i : i + batch_size]
        for i in range(0, len(player_list), batch_size)
    ]
    total_batches = len(batches)
    all_player_records = []

    print(
        f"Processing {len(player_list)} players across {total_batches} batches..."
    )

    for idx, batch in enumerate(batches):
        batch_num = idx + 1
        prompt = PROMPT_TEMPLATE.replace("{batch_players}", json.dumps(batch))

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response_text = llm_api(prompt)
                cleaned_text = re.sub(
                    r"^```(?:json)?\s*|\s*```$", "", response_text.strip()
                )
                data = json.loads(cleaned_text)
                all_player_records.extend(data)
                print(f"✓ Completed batch {batch_num}/{total_batches}")
                break
            except Exception as e:
                print(
                    f"⚠️ Batch {batch_num} attempt {attempt+1} failed: {e}. Retrying in 20s..."
                )
                time.sleep(20)

        # Pause to comply with 5 Requests-Per-Minute limit
        if batch_num < total_batches:
            time.sleep(delay_seconds)

    metadata_df = pd.DataFrame(all_player_records)
    metadata_df.to_csv("player_metadata.csv", index=False)
    print(
        f"\nDone! Saved {len(metadata_df)} players to 'player_metadata.csv'."
    )
    return metadata_df


# Run sequentially with 100 players per batch
player_meta_df = generate_player_metadata_robust(
    unique_players, batch_size=100, delay_seconds=12
)

Processing 806 players across 9 batches...
✓ Completed batch 1/9
✓ Completed batch 2/9
✓ Completed batch 3/9
✓ Completed batch 4/9
✓ Completed batch 5/9
✓ Completed batch 6/9
✓ Completed batch 7/9
✓ Completed batch 8/9
✓ Completed batch 9/9

Done! Saved 806 players to 'player_metadata.csv'.


In [77]:
import numpy as np
import pandas as pd

# 1. Load generated metadata lookup table
player_meta = pd.read_csv("player_metadata.csv")

# Clean up any potential duplicate columns if re-running
df_main = df_main.drop(
    columns=["batter_hand", "bowler_cat", "matchup_type"], errors="ignore"
)

# 2. Merge Batter Attributes (batting_hand)
df_main = df_main.merge(
    player_meta[["name", "batting_hand"]].rename(
        columns={"name": "batter", "batting_hand": "batter_hand"}
    ),
    on="batter",
    how="left",
)

# 3. Merge Bowler Attributes (bowling_style -> bowler_cat)
df_main = df_main.merge(
    player_meta[["name", "bowling_style"]].rename(
        columns={"name": "bowler", "bowling_style": "bowler_cat"}
    ),
    on="bowler",
    how="left",
)

# 4. Fill missing values with standard cricket defaults
df_main["batter_hand"] = df_main["batter_hand"].fillna("RHB")
df_main["bowler_cat"] = df_main["bowler_cat"].fillna("Unknown")

# 5. Construct 5-Category Tactical Matchup Feature
matchup_conditions = [
    # Pace Same Arm
    (
        (df_main["batter_hand"] == "RHB") & (df_main["bowler_cat"] == "RA_Pace")
    )
    | (
        (df_main["batter_hand"] == "LHB") & (df_main["bowler_cat"] == "LA_Pace")
    ),
    # Pace Opposite Arm
    (
        (df_main["batter_hand"] == "RHB") & (df_main["bowler_cat"] == "LA_Pace")
    )
    | (
        (df_main["batter_hand"] == "LHB") & (df_main["bowler_cat"] == "RA_Pace")
    ),
    # Finger Spin Turning In
    (
        (df_main["batter_hand"] == "RHB")
        & (df_main["bowler_cat"] == "RA_Offspin")
    )
    | (
        (df_main["batter_hand"] == "LHB")
        & (df_main["bowler_cat"] == "LA_Orthodox")
    ),
    # Finger Spin Turning Away
    (
        (df_main["batter_hand"] == "RHB")
        & (df_main["bowler_cat"] == "LA_Orthodox")
    )
    | (
        (df_main["batter_hand"] == "LHB")
        & (df_main["bowler_cat"] == "RA_Offspin")
    ),
    # Wrist Spin (Leg-Spin & Chinaman dual-threat)
    (df_main["bowler_cat"] == "Wrist_Spin"),
]

matchup_choices = [
    "Pace_Same_Arm",
    "Pace_Opposite_Arm",
    "Finger_Spin_Turning_In",
    "Finger_Spin_Turning_Away",
    "Wrist_Spin",
]

df_main["matchup_type"] = np.select(
    matchup_conditions, matchup_choices, default="Unknown"
)